# Modeling Single Subject Data

> Adapted from [DartBrains](https://dartbrains.org/) by Luke Chang (© Luke Chang, supported by NSF CAREER Award 1848370), used under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). **Changes made:** trimmed to the exercise sections, setup code reduced to the cells the exercises depend on, display-only calls removed. Exercise solutions are my own work. This notebook is therefore also distributed under CC BY-SA 4.0. No endorsement by the original author is implied.


In [1]:
%matplotlib inline

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nibabel as nib
from nltools.file_reader import onsets_to_dm
from nltools.stats import regress, zscore
from nltools.data import Brain_Data, Design_Matrix
from nltools.stats import find_spikes 
from nilearn.plotting import view_img, glass_brain, plot_stat_map
from bids import BIDSLayout, BIDSValidator
from IPython.display import Image

data_dir = '/home/jovyan/shared/PSYCH60-22S/data/localizer'
layout = BIDSLayout(data_dir, derivatives=True)


/opt/conda/lib/python3.9/site-packages/nilearn/input_data/__init__.py:27: FutureWarning: The import path 'nilearn.input_data' is deprecated in version 0.9. Importing from 'nilearn.input_data' will be possible at least until release 0.13.0. Please import from 'nilearn.maskers' instead.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.9/site-packages/bids/layout/validation.py:151: UserWarning: The PipelineDescription field was superseded by GeneratedBy in BIDS 1.4.0. You can use ``pybids upgrade`` to update your derivative dataset.
  warnings.warn("The PipelineDescription field was superseded "


In [ ]:
def load_bids_events(layout, subject):
    """Create a design_matrix instance from BIDS event file"""
    tr = layout.get_tr()
    boldFiles = layout.get(subject=subject, scope='derivatives', suffix='bold', return_type='filename', extension='nii.gz')
    preprocFile = [FILE for i, FILE in enumerate(boldFiles) if FILE[-19:] == 'preproc_bold.nii.gz']
    n_tr = nib.load(preprocFile[0]).shape[3]
    onsets = pd.read_csv(layout.get(subject=subject, suffix='events')[0].path, sep='\t')
    onsets.columns = ['Onset', 'Duration', 'Stim']
    return (onsets_to_dm(onsets, sampling_freq=1 / tr, run_length=n_tr), preprocFile)
dm, preprocFile = load_bids_events(layout, 'S01')

In [ ]:
dm_conv = dm.convolve()

In [ ]:
dm_conv_filt = dm_conv.add_dct_basis(duration=128)

In [ ]:
dm_conv_filt_poly = dm_conv_filt.add_poly(order=3, include_lower=True)

In [ ]:
data = Brain_Data(preprocFile)

In [ ]:
covariates = pd.read_csv(layout.get(subject='S01', scope='derivatives', extension='.tsv')[0].path, sep='\t')
mc = covariates[['trans_x', 'trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z']]

In [ ]:
def make_motion_covariates(mc, tr):
    z_mc = zscore(mc)
    all_mc = pd.concat([z_mc, z_mc ** 2, z_mc.diff(), z_mc.diff() ** 2], axis=1)
    all_mc.fillna(value=0, inplace=True)
    return Design_Matrix(all_mc, sampling_freq=1 / tr)
tr = layout.get_tr()
mc_cov = make_motion_covariates(mc, tr)

In [ ]:
spikes = data.find_spikes(global_spike_cutoff=2.5, diff_spike_cutoff=2.5)
f, a = plt.subplots(figsize=(15, 3))
spikes = Design_Matrix(spikes.iloc[:, 1:], sampling_freq=1 / tr)

In [ ]:
dm_conv_filt_poly_cov = pd.concat([dm_conv_filt_poly, mc_cov, spikes], axis=1)

In [ ]:
fwhm = 6
smoothed = data.smooth(fwhm=fwhm)

In [ ]:
smoothed.X = dm_conv_filt_poly_cov
stats = smoothed.regress()

## Exercises

### 1. Which regions are more involved with visual compared to auditory sensory processing?

In [40]:
# get indices of betas to contrast (run dm.info() to get an easy-to-parse list)
vInds = [0, 1, 4, 6, 7, 9] # video indices
aInds = [2, 3, 5, 8] # audio indices

# initialize contrast vector
c1 = np.zeros(len(stats['beta']))
c1[vInds] = 1/len(vInds)
c1[aInds] = -1/len(aInds)
c1

vva = stats['beta'] * c1

vva.iplot()

vva.write('exercise_1_vva.nii.gz')


interactive(children=(FloatText(value=0.0, description='Threshold'), HTML(value='Image is 3D', description='Vo…

In [41]:
dm.info()

<class 'nltools.data.design_matrix.Design_Matrix'>
RangeIndex: 128 entries, 0 to 127
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   video_computation        128 non-null    float64
 1   horizontal_checkerboard  128 non-null    float64
 2   audio_right_hand         128 non-null    float64
 3   audio_sentence           128 non-null    float64
 4   video_right_hand         128 non-null    float64
 5   audio_left_hand          128 non-null    float64
 6   video_left_hand          128 non-null    float64
 7   vertical_checkerboard    128 non-null    float64
 8   audio_computation        128 non-null    float64
 9   video_sentence           128 non-null    float64
dtypes: float64(10)
memory usage: 10.1 KB


### 2. Which regions are more involved in processing numbers compared to words?

In [42]:
cInds = [0, 8]
wInds = [3, 9]

# initialize contrast vector
c1 = np.zeros(len(stats['beta']))
c1[cInds] = 1/len(cInds)
c1[wInds] = -1/len(wInds)
c1

cvw = stats['beta'] * c1

cvw.iplot()

cvw.write('exercise_2_cvw.nii.gz')


interactive(children=(FloatText(value=0.0, description='Threshold'), HTML(value='Image is 3D', description='Vo…

### 3. Which regions are more involved with motor compared to cognitive processes (e.g., language and math)?

In [ ]:
mInds = [2,4,5,6]
cInds = [0,3,8,9]

# initialize contrast vector
c1 = np.zeros(len(stats['beta']))
c1[mInds] = 1/len(mInds)
c1[cInds] = -1/len(cInds)
c1

mvc = stats['beta'] * c1

mvc.iplot()

mvc.write('exercise_3_mvc.nii.gz')

interactive(children=(FloatText(value=0.0, description='Threshold'), HTML(value='Image is 3D', description='Vo…

### 4. How are your results impacted by different smoothing kernels?

In [ ]:
fwhm=[2, 10]

# initialize contrast vector
c1 = np.zeros(len(stats['beta']))

c1[1] = 1 # horizontal
c1[7] = -1 # vert

for FWHM in fwhm:
    
    # smooth
    smoothed = data.smooth(fwhm=FWHM)
    
    # add design matrix and regress
    smoothed.X = dm_conv_filt_poly_cov
    stats = smoothed.regress()
    
    # apply contrast
    hvv = stats['beta'] * c1
    
    # plot
    hvv.iplot()
    
    # write
    hvv.write(f'exercise_4_hvv_fwhm{FWHM}.nii.gz')


